# 01 线性回归 Linear Regression

依赖安装说明：`pip install numpy matplotlib scikit-learn`

线性回归是最基础的监督学习模型：输入一组特征 `X`，预测一个连续数值 `y`。它适合回答“某个变量大概会是多少”的问题，例如房价、销量、温度、成本。

本 notebook 会从一条直线开始，讲清楚模型、loss、梯度下降和 `sklearn` 实战。


## 0. 学习目标和阅读地图

学完这一节你应该能回答四个问题：

1. 为什么线性回归是在找一条“平均误差最小”的直线或超平面。
2. 为什么平方误差会让大错误受到更重惩罚。
3. 梯度下降里的 `w`、`b`、`grad_w`、`grad_b` 分别对应什么。
4. 什么时候线性回归足够好，什么时候应该换成非线性模型或加特征工程。

建议先把一维情况 `y = wx + b` 看懂，再把 `x` 想成向量。多维线性回归没有本质变化，只是把一个斜率 `w` 变成了一组权重。


## 1. 模型解决什么问题

线性回归假设输出可以由输入特征的线性组合解释：

$$\hat y = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b$$

如果只有一个特征，就是一条直线：

$$\hat y = wx + b$$

学习的目标是找到合适的 `w` 和 `b`，让预测值 `\hat y` 尽量接近真实值 `y`。


## 2. 数学逻辑

最常用的损失函数是均方误差 Mean Squared Error：

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat y_i)^2$$

训练就是最小化这个损失：

$$\min_{w,b} \frac{1}{n}\sum_{i=1}^{n}(y_i - (wx_i+b))^2$$

梯度下降的更新形式是：

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}$$

$$b \leftarrow b - \eta \frac{\partial L}{\partial b}$$

其中 `eta` 是学习率，控制每一步走多大。


## 2.1 推导拆开看：从误差到梯度

单个样本的预测误差是：

$$e_i = \hat y_i - y_i = wx_i + b - y_i$$

整个数据集的 MSE 是：

$$L = \frac{1}{n}\sum_i e_i^2$$

对 `w` 求导：

$$\frac{\partial L}{\partial w}=\frac{2}{n}\sum_i e_i x_i$$

对 `b` 求导：

$$\frac{\partial L}{\partial b}=\frac{2}{n}\sum_i e_i$$

直觉上，如果某个点 `x_i` 很大，而且模型预测高了，那么 `e_i x_i` 会把 `w` 往下降；如果整体都预测高了，`grad_b` 会把截距往下降。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

np.random.seed(42)

X = np.linspace(-3, 3, 80).reshape(-1, 1)
y = 2.5 * X[:, 0] + 1.2 + np.random.normal(0, 1.0, size=len(X))

plt.scatter(X[:, 0], y, s=24)
plt.title('合成数据：大致是一条带噪声的直线')
plt.xlabel('x')
plt.ylabel('y')
plt.show()


## 2.2 数据流和变量形状

这一节的核心变量：

- `X.shape == (80, 1)`：80 个样本，每个样本 1 个特征。
- `y.shape == (80,)`：80 个真实连续值。
- `w`：斜率，一个标量。
- `b`：截距，一个标量。
- `y_hat`：模型对每个样本的预测。
- `error = y_hat - y`：每个样本的残差。

真实项目里，`X` 往往是 `(样本数, 特征数)`，例如房屋面积、楼层、地段编码等。线性回归会给每个特征一个权重，权重正负表示该特征和目标的线性关系方向。


In [ ]:
# 从零实现：只拟合一维线性回归 y_hat = w*x + b
w = 0.0
b = 0.0
lr = 0.05
loss_history = []

x_flat = X[:, 0]
for step in range(300):
    y_hat = w * x_flat + b
    error = y_hat - y
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    grad_w = 2 * np.mean(error * x_flat)
    grad_b = 2 * np.mean(error)
    w -= lr * grad_w
    b -= lr * grad_b

print('从零训练得到的 w:', round(w, 3))
print('从零训练得到的 b:', round(b, 3))
print('最终 MSE:', round(loss_history[-1], 3))

plt.plot(loss_history)
plt.title('梯度下降过程中 MSE 下降')
plt.xlabel('step')
plt.ylabel('MSE')
plt.show()


## 2.3 从零实现代码怎么读

从零实现的训练循环可以按 5 行逻辑理解：

1. `y_hat = w * x_flat + b`：用当前参数预测。
2. `error = y_hat - y`：看每个点错了多少。
3. `loss = np.mean(error ** 2)`：把错误汇总成一个数字。
4. `grad_w / grad_b`：计算参数应该往哪个方向动。
5. `w -= lr * grad_w`：沿着让 loss 下降的方向更新。

这就是大多数机器学习训练循环的雏形：预测、算损失、算梯度、更新参数。


In [ ]:
# sklearn 实战：真实项目里通常用库完成拟合、预测和评估
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)
print('sklearn w:', round(model.coef_[0], 3))
print('sklearn b:', round(model.intercept_, 3))
print('测试集 MSE:', round(mean_squared_error(y_test, pred), 3))
print('测试集 R^2:', round(r2_score(y_test, pred), 3))

line_x = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.scatter(X[:, 0], y, s=24, label='data')
plt.plot(line_x[:, 0], model.predict(line_x), color='red', label='fit')
plt.legend()
plt.title('线性回归拟合结果')
plt.show()


In [ ]:
# 诊断：残差图可以帮助判断线性假设是否合理
full_pred = model.predict(X)
residuals = y - full_pred

plt.scatter(full_pred, residuals, s=24)
plt.axhline(0, color='red', linestyle='--')
plt.title('残差图：好的线性模型残差应围绕 0 随机分布')
plt.xlabel('预测值')
plt.ylabel('残差 y - y_hat')
plt.show()

print('残差均值:', round(residuals.mean(), 4))
print('残差标准差:', round(residuals.std(), 4))


## 3.1 如何诊断线性回归

残差图非常重要。如果残差围绕 0 随机散开，线性模型通常还可以；如果残差呈现 U 形、波浪形或漏斗形，就说明模型漏掉了结构。

常见诊断信号：

- 残差有曲线形状：需要非线性特征，例如 `x^2`。
- 残差随预测值变大而变散：可能存在异方差。
- 少数点残差极大：可能有异常值，MSE 会被它们强烈影响。


## 3. 评价指标

- `MSE`：误差平方的平均值，越低越好，对大误差很敏感。
- `MAE`：绝对误差的平均值，更容易按原单位理解。
- `R^2`：解释方差比例，越接近 1 越好；小于 0 通常说明模型还不如直接预测均值。

## 4. 常见误区

- 线性回归不等于只能拟合直线；加入多项式特征后也能拟合曲线，但模型对参数仍是线性的。
- 特征尺度差异很大时，梯度下降会变慢；库的闭式解不一定受同样影响。
- 相关不等于因果。线性回归能做预测，不自动证明因果关系。

## 5. 小实验

- 把噪声标准差从 `1.0` 改成 `3.0`，观察 MSE 和拟合线。
- 把学习率 `lr` 改大，看 loss 是否震荡。
- 把样本数从 `80` 改成 `10`，观察过拟合和不稳定性。


## 6. 复习清单

- 线性回归学习的是权重和截距，不是记住样本。
- MSE 的平方会放大大误差。
- 梯度下降每一步都在问：参数怎么动能让 loss 变小？
- `R^2` 高不等于因果成立，也不等于未来一定预测好。
- 看回归模型时，至少同时看指标、残差图和数据分布。
